# ⚡ Notebook 03 — Model Training: LightGBM + SynapseML

**Goal:** Train a LightGBM model using Fabric's native SynapseML library. LightGBM typically outperforms scikit-learn models on tabular data.

> **Run time:** ~5 min

In [ ]:
# Import MLflow, LightGBM, and evaluation helpers for gradient boosting training
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import lightgbm as lgb

# Load Feature Store
# Load the engineered credit risk feature store into pandas for LightGBM
df = spark.table('silver_credit_risk_features').toPandas()
# Confirm the dataset size before preparing LightGBM inputs
print(f'Loaded {len(df)} records')


## Step 1 — Prepare Features (LightGBM handles categoricals natively)

In [ ]:
# Cast categorical borrower fields to pandas category dtype so LightGBM can use them natively
cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    df[col] = df[col].astype('category')

# Define the numeric and categorical predictors used by the LightGBM model
feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType','LoanPurpose','EmploymentType','HomeOwnership'
]

# Build the feature matrix and binary default target for training
X = df[feature_cols].fillna(0)
y = df['IsDefault'].astype(int)
# Split the data 80/20 into stratified training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Verify the training and test sample sizes before fitting the model
print(f'Train: {len(X_train)} | Test: {len(X_test)}')


## Step 2 — Train LightGBM with MLflow Autolog

In [ ]:
# Point tracking to the shared MLflow experiment and enable LightGBM autologging
mlflow.set_experiment('CreditRiskScoring')
mlflow.lightgbm.autolog()  # Automatically logs params, metrics, and model!

# Start an MLflow run for the LightGBM credit risk model
with mlflow.start_run(run_name='LightGBM'):
# Define the LightGBM hyperparameters for binary default prediction
    params = {
        'objective':        'binary',
        'metric':           'auc',
        'learning_rate':    0.05,
        'num_leaves':       63,
        'max_depth':        -1,
        'min_child_samples': 20,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'n_estimators':     500,
        'random_state':     42,
        'verbose':          -1
    }

# Train the LightGBM model with early stopping against the validation set
    model_lgb = lgb.LGBMClassifier(**params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

# Score the test set and calculate AUC-ROC and average precision for LightGBM
    y_prob_lgb = model_lgb.predict_proba(X_test)[:, 1]
    auc_lgb    = roc_auc_score(y_test, y_prob_lgb)
    ap_lgb     = average_precision_score(y_test, y_prob_lgb)

# Log the key ranking metrics for the LightGBM run to MLflow
    mlflow.log_metric('auc_roc', auc_lgb)
    mlflow.log_metric('avg_precision', ap_lgb)

# Print detailed classification performance for the LightGBM model
    print(f'\n✅ LightGBM  |  AUC-ROC: {auc_lgb:.4f}  |  Avg Precision: {ap_lgb:.4f}')
    print(classification_report(y_test, model_lgb.predict(X_test), target_names=['No Default','Default']))


## Step 3 — Feature Importance (Top 10)

In [ ]:
import pandas as pd

# Rank the strongest predictors using the fitted LightGBM feature importances
fi = pd.DataFrame({
    'Feature':   feature_cols,
    'Importance': model_lgb.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

# Display the top feature importance results for model interpretation
print('Top 10 Features by Importance (LightGBM):')
print(fi.to_string(index=False))


## Step 4 — Cross-Validation (5-fold)

In [ ]:
# Import cross-validation helpers to test LightGBM stability across folds
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

# Configure a stratified 5-fold split to preserve default-rate balance in every fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Run 5-fold cross-validation and score each fold using AUC-ROC
cv_scores = cross_val_score(clone(model_lgb), X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

# Summarize the average and per-fold cross-validation performance
print(f'5-Fold CV AUC-ROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Individual folds: {[round(s,4) for s in cv_scores]}')
